In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [6]:
CSV_PATH = "/kaggle/input/asl-signs/train.csv"
PARQUET_ROOT = "/kaggle/input/asl-signs"

df = pd.read_csv(CSV_PATH)
print("Total samples:", len(df))
print("Total classes:", df['sign'].nunique())


Total samples: 94477
Total classes: 250


In [5]:
sign_to_frames = {}

for sign, group in tqdm(df.groupby("sign")):
    frame_counts = []
    
    for path in group["path"].values:
        full_path = os.path.join(PARQUET_ROOT, path)
        pq = pd.read_parquet(full_path, columns=["frame"])
        frame_counts.append(pq["frame"].nunique())
    
    sign_to_frames[sign] = int(np.median(frame_counts))


100%|██████████| 250/250 [27:21<00:00,  6.56s/it]


In [8]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans

In [9]:
def compute_motion_score(parquet_path):
    """
    Motion = mean frame-to-frame landmark displacement
    """
    df = pd.read_parquet(parquet_path)

    if "frame" not in df.columns:
        return 0.0, 0

    # Landmark columns
    coord_cols = [c for c in df.columns if c not in ["frame", "row_id"]]
    frames = df["frame"].unique()

    if len(frames) < 2:
        return 0.0, len(frames)

    df = df.sort_values("frame")
    landmarks = df[coord_cols].values

    diffs = np.diff(landmarks, axis=0)
    motion = np.mean(np.linalg.norm(diffs, axis=1))

    return motion, len(frames)


In [10]:
def analyze_dataset(train_csv_path, base_path):
    train_df = pd.read_csv(train_csv_path)

    records = []

    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        parquet_path = f"{base_path}/{row['path']}"

        try:
            motion, frames = compute_motion_score(parquet_path)
        except:
            motion, frames = 0.0, 0

        records.append({
            "sign": row["sign"],
            "motion_score": motion,
            "num_frames": frames
        })

    return pd.DataFrame(records)


In [11]:
TRAIN_CSV = "/kaggle/input/asl-signs/train.csv"
BASE_PATH = "/kaggle/input/asl-signs"
SAMPLE_SIZE = 8000      # reduce if kernel times out
N_MODELS = 4
RANDOM_STATE = 42


In [12]:
# Load CSV
df = pd.read_csv(TRAIN_CSV)

# Optional speed-up
df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

# Save temp CSV
df.to_csv("tmp_train.csv", index=False)

# Analyze motion
motion_df = analyze_dataset("tmp_train.csv", BASE_PATH)

print("Motion extraction done")

# Aggregate PER SIGN
sign_stats = (
    motion_df
    .groupby("sign")
    .agg(
        median_motion=("motion_score", "median"),
        mean_frames=("num_frames", "mean"),
        samples=("motion_score", "count")
    )
    .reset_index()
)

print("Unique signs:", len(sign_stats))


100%|██████████| 8000/8000 [04:07<00:00, 32.35it/s]

Motion extraction done
Unique signs: 250


In [13]:
kmeans = KMeans(n_clusters=N_MODELS, random_state=RANDOM_STATE)

sign_stats["model_id"] = kmeans.fit_predict(
    sign_stats["median_motion"].values.reshape(-1, 1)
)

cluster_to_signs = {}

for m in range(N_MODELS):
    signs = sign_stats[sign_stats["model_id"] == m]["sign"].tolist()
    cluster_to_signs[m] = signs

    print(f"\nModel {m}")
    print(f"  #Classes: {len(signs)}")
    print(f"  Sample signs: {signs[:10]}")



Model 0
  #Classes: 250
  Sample signs: ['TV', 'after', 'airplane', 'all', 'alligator', 'animal', 'another', 'any', 'apple', 'arm']

Model 1
  #Classes: 0
  Sample signs: []

Model 2
  #Classes: 0
  Sample signs: []

Model 3
  #Classes: 0
  Sample signs: []


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [14]:
print(sign_stats[["median_motion", "mean_frames", "samples"]].describe())


       median_motion  mean_frames    samples
count          250.0        250.0  250.00000
mean             0.0          0.0   32.00000
std              0.0          0.0    5.46842
min              0.0          0.0   19.00000
25%              0.0          0.0   29.00000
50%              0.0          0.0   32.00000
75%              0.0          0.0   35.00000
max              0.0          0.0   53.00000


In [15]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("/kaggle/input/asl-signs/train.csv")

all_signs = sorted(train_df["sign"].unique())
print("Total signs:", len(all_signs))  # should be 250


Total signs: 250


In [16]:
NUM_MODELS = 4

splits = np.array_split(all_signs, NUM_MODELS)

cluster_to_signs = {
    i: splits[i].tolist() for i in range(NUM_MODELS)
}


In [17]:
for i, signs in cluster_to_signs.items():
    print(f"\nModel {i}")
    print(f"  #Classes: {len(signs)}")
    print(f"  Sample signs: {signs[:10]}")



Model 0
  #Classes: 63
  Sample signs: ['TV', 'after', 'airplane', 'all', 'alligator', 'animal', 'another', 'any', 'apple', 'arm']

Model 1
  #Classes: 63
  Sample signs: ['drink', 'drop', 'dry', 'dryer', 'duck', 'ear', 'elephant', 'empty', 'every', 'eye']

Model 2
  #Classes: 62
  Sample signs: ['jeans', 'jump', 'kiss', 'kitty', 'lamp', 'later', 'like', 'lion', 'lips', 'listen']

Model 3
  #Classes: 62
  Sample signs: ['room', 'sad', 'same', 'say', 'scissors', 'see', 'shhh', 'shirt', 'shoe', 'shower']
